In [ ]:
import mlflow
import pandas as pd

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("House Price Prediction Experiment")

In [3]:
house_sales = pd.read_csv("../house_sales.csv")

In [6]:
house_sales.describe()

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580.000000,1.358000e+04,13580.000000,13580.000000,13580.000000,13580.000000,13518.000000,13580.000000,7130.000000,8205.000000,13580.000000,13580.000000,13580.000000
mean,2.937997,1.075684e+06,10.137776,3105.301915,2.914728,1.534242,1.610075,558.416127,151.967650,1964.684217,-37.809203,144.995216,7454.417378
std,0.955748,6.393107e+05,5.868725,90.676964,0.965921,0.691712,0.962634,3990.669241,541.014538,37.273762,0.079260,0.103916,4378.581772
min,1.000000,8.500000e+04,0.000000,3000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1196.000000,-38.182550,144.431810,249.000000
25%,2.000000,6.500000e+05,6.100000,3044.000000,2.000000,1.000000,1.000000,177.000000,93.000000,1940.000000,-37.856822,144.929600,4380.000000
50%,3.000000,9.030000e+05,9.200000,3084.000000,3.000000,1.000000,2.000000,440.000000,126.000000,1970.000000,-37.802355,145.000100,6555.000000
75%,3.000000,1.330000e+06,13.000000,3148.000000,3.000000,2.000000,2.000000,651.000000,174.000000,1999.000000,-37.756400,145.058305,10331.000000
max,10.000000,9.000000e+06,48.100000,3977.000000,20.000000,8.000000,10.000000,433014.000000,44515.000000,2018.000000,-37.408530,145.526350,21650.000000


In [8]:
house_sales = house_sales.dropna(axis=0)
house_sales.describe()

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,6196.000000,6.196000e+03,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000
mean,2.931407,1.068828e+06,9.751097,3101.947708,2.902034,1.576340,1.573596,471.006940,141.568645,1964.081988,-37.807904,144.990201,7435.489509
std,0.971079,6.751564e+05,5.612065,86.421604,0.970055,0.711362,0.929947,897.449881,90.834824,38.105673,0.075850,0.099165,4337.698917
min,1.000000,1.310000e+05,0.000000,3000.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1196.000000,-38.164920,144.542370,389.000000
25%,2.000000,6.200000e+05,5.900000,3044.000000,2.000000,1.000000,1.000000,152.000000,91.000000,1940.000000,-37.855438,144.926198,4383.750000
50%,3.000000,8.800000e+05,9.000000,3081.000000,3.000000,1.000000,1.000000,373.000000,124.000000,1970.000000,-37.802250,144.995800,6567.000000
75%,4.000000,1.325000e+06,12.400000,3147.000000,3.000000,2.000000,2.000000,628.000000,170.000000,2000.000000,-37.758200,145.052700,10175.000000
max,8.000000,9.000000e+06,47.400000,3977.000000,9.000000,8.000000,10.000000,37000.000000,3112.000000,2018.000000,-37.457090,145.526350,21650.000000


In [ ]:
features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longtitude']
X = house_sales[features]
y = house_sales["Price"]
print(X.shape)
print(y.shape)

In [20]:
from sklearn.model_selection import train_test_split

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)

In [24]:
# Fist Model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

with mlflow.start_run(run_name = "Linear Regression Model"):
    lr_model_house_sales = LinearRegression()
    lr_model_house_sales.fit(X_train, y_train)

    y_preds_lr = lr_model_house_sales.predict(X_test)
    mae_lr = mean_absolute_error(y_test, y_preds_lr)
    print(f"Mean Absolute Error (Linear Regression Model): {mae_lr}")

Mean Absolute Error: 293237.74374876515


In [26]:
# Second Model
from sklearn.tree import DecisionTreeRegressor

with mlflow.start_run(run_name = "Decision Tree Regressor Model"):
    dt_model_house_sales = DecisionTreeRegressor(random_state=0)
    dt_model_house_sales.fit(X_train, y_train)

    y_preds_dt = dt_model_house_sales.predict(X_test)
    mae_dt = mean_absolute_error(y_test, y_preds_dt)
    print(f"Mean Absolute Error (Decision Tree Regressor Model): {mae_dt}")

Mean Absolute Error (Decision Tree Regressor Model): 248510.6655907037


In [ ]:
# Try different max_leaf_nodes (Underfitting and Overfitting)

In [30]:
def get_mae(max_leaf_nodes, X_train, X_test, y_train, y_test):
    with mlflow.start_run(run_name = f"Decision Tree Regressor Model - Max leaf nodes: {max_leaf_nodes}"):
        model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=0)
        model.fit(X_train, y_train)
        y_preds = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_preds)
    return mae

In [31]:
# compare MAE with differing values of max_leaf_nodes
for max_leaf_nodes in [5, 50, 500, 5000]:
    mae = get_mae(max_leaf_nodes, X_train, X_test, y_train, y_test)
    print(f"Max leaf nodes: {max_leaf_nodes} \t\t Mean Absolute Error: {mae}")

Max leaf nodes: 5 		 Mean Absolute Error: 324110.91454760684
Max leaf nodes: 50 		 Mean Absolute Error: 252108.93840069315
Max leaf nodes: 500 		 Mean Absolute Error: 239204.22623333166
Max leaf nodes: 5000 		 Mean Absolute Error: 249358.76436410588


In [32]:
# Third Model
from sklearn.ensemble import RandomForestRegressor

with mlflow.start_run(run_name = "Random Forest Regressor"):
    rf_model_house_sales = RandomForestRegressor()
    rf_model_house_sales.fit(X_train, y_train)

    y_preds_rf = rf_model_house_sales.predict(X_test)
    mae_rf = mean_absolute_error(y_test, y_preds_rf)
    print(f"Mean Absolute Error (Random Forest Model): {mae_rf}")

Mean Absolute Error (Random Forest Model): 175227.76966645147
